# Activation Additions (LLAMA 3 8b)


For running on Google Colab, change **Runtime -> GPU with High Ram**.

For LLAMA3-8b you may need to use a HuggingFace api key.

Some parts of the code build upon `https://github.com/jonnypei/acl23-preadd.git`

## Misc

In [ ]:
# !pip install transformer_lens
# !pip install sentence_transformers==2.7.0
# !pip install tqdm

# !pip install optimum
# !pip install onnxruntime
# !pip install onnx
# !pip install transformers sentencepiece --quiet

# !pip install datasets
!python --version
!pip list

Package                       Version
----------------------------- -----------
accelerate                    1.14.0
aiohappyeyeballs              2.6.2
aiohttp                       3.14.1
aiosignal                     1.4.0
annotated-doc                 0.0.4
annotated-types               0.7.0
anyio                         4.14.1
argon2-cffi                   25.1.0
argon2-cffi-bindings          25.1.0
arrow                         1.4.0
asttokens                     3.0.1
async-lru                     2.3.0
attrs                         26.1.0
babel                         2.18.0
beartype                      0.22.9
beautifulsoup4                4.15.0
better-abc                    0.0.3
bleach                        6.4.0
certifi                       2026.6.17
cffi                          2.0.0
charset-normalizer            3.4.7
click                         8.4.2
comm                          0.2.3
datasets                      5.0.0
debugpy                       1.8.21
decora

In [3]:
from sentence_transformers import SentenceTransformer
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

from datasets import load_dataset, concatenate_datasets

# from optimum.onnxruntime import ORTModelForSequenceClassification
from sklearn.metrics.pairwise import cosine_similarity

# import openai
# from googleapiclient import discovery

import requests
import json
import random
import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy.stats import ttest_rel


from typing import Dict, Union, List, Any
from tqdm import tqdm
import os
import re
import time

import transformers
print("transformers: ", transformers.__version__)
print("torch: ", torch.__version__)

transformers:  4.57.6
torch:  2.7.1+cu126


In [4]:
## params and directories
# save_dir = "/content/drive/MyDrive/actadd_reb" # change or create such dir
prompts_setting = "llama3_sentiment"
display = True
get_x_vector_preset = "actadd"

sample_n = 10

prompt_add, prompt_sub = "Intent to praise", "Intent to hurt"
SEED = 0
sampling_kwargs = dict(temperature=1.0, top_p=0.3, freq_penalty=1.0)


In [ ]:
# Load LLAMA3-8b
model_name = "/scratch/common_models/gpt2/"  # "meta-llama/Meta-Llama-3-8B"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def initialize_model(model_name: str = "/scratch/common_models/gpt2/", device: str = None) -> torch.nn.Module:
    print(f"model {model_name} initialized")
    torch.set_grad_enabled(False)
    model = HookedTransformer.from_pretrained(model_name)  # only accepts official model names, not path
    model.eval()
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Decive being used is {device}")
    model.to(device)
    return model

model_llama = initialize_model(model_name, device)

model /scratch/common_models/gpt2/ initialized


ValueError: /scratch/common_models/gpt2/ not found. Valid official model names (excl aliases): ['01-ai/Yi-34B', '01-ai/Yi-34B-Chat', '01-ai/Yi-6B', '01-ai/Yi-6B-Chat', 'ai-forever/mGPT', 'allenai/OLMo-1B-hf', 'allenai/OLMo-2-0425-1B', 'allenai/OLMo-2-1124-7B', 'allenai/Olmo-3-32B-Think', 'allenai/Olmo-3-7B-Instruct', 'allenai/Olmo-3-7B-Think', 'allenai/Olmo-3.1-32B-Instruct', 'allenai/Olmo-3.1-32B-Think', 'allenai/OLMo-7B-hf', 'allenai/OLMoE-1B-7B-0924', 'ArthurConmy/redwood_attn_2l', 'Baidicoot/Othello-GPT-Transformer-Lens', 'bigcode/santacoder', 'bigscience/bloom-1b1', 'bigscience/bloom-1b7', 'bigscience/bloom-3b', 'bigscience/bloom-560m', 'bigscience/bloom-7b1', 'codellama/CodeLlama-7b-hf', 'codellama/CodeLlama-7b-Instruct-hf', 'codellama/CodeLlama-7b-Python-hf', 'distilgpt2', 'EleutherAI/gpt-j-6B', 'EleutherAI/gpt-neo-1.3B', 'EleutherAI/gpt-neo-125M', 'EleutherAI/gpt-neo-2.7B', 'EleutherAI/gpt-neox-20b', 'EleutherAI/pythia-1.4b', 'EleutherAI/pythia-1.4b-deduped', 'EleutherAI/pythia

In [ ]:
# ActAddd logic

def prepare_prompts(prompt_add: str, prompt_sub: str, model: torch.nn.Module) -> tuple:
    def tlen(prompt): return model.to_tokens(prompt).shape[1]
    def pad_right(prompt, length): return prompt + " " * (length - tlen(prompt))
    print("tlen(prompt_add): ", tlen(prompt_add), " tlen(prompt_sub): ", tlen(prompt_sub))
    l = max(tlen(prompt_add), tlen(prompt_sub))
    return pad_right(prompt_add, l), pad_right(prompt_sub, l)

def get_resid_pre(prompt: str, layer: int, model: torch.nn.Module) -> torch.Tensor:
    name = f"blocks.{layer}.hook_resid_pre"
    cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n == name)
    with model.hooks(fwd_hooks=caching_hooks):
        _ = model(prompt)
    return cache[name]

def ave_hook(resid_pre, hook, act_diff, coeff):
    if resid_pre.shape[1] == 1: return
    # print(f"resid_pre.shape: {resid_pre.shape}, act_diff: {act_diff.shape}")
    ppos, apos = resid_pre.shape[1], act_diff.shape[1]
    assert apos <= ppos, f"More mod tokens ({apos}) than prompt tokens ({ppos})!"  ####
    resid_pre[:, :apos, :] += coeff * act_diff
    print("resid_pre: ", resid_pre.shape, resid_pre)

def hooked_generate(prompt_batch: List[str], editing_hooks: list, seed: int, model: torch.nn.Module, **kwargs) -> torch.Tensor:
    if seed is not None: torch.manual_seed(seed)
    with model.hooks(fwd_hooks=editing_hooks):
        print(f"prompt_batch: {prompt_batch}")
        tokenized = model.to_tokens(prompt_batch)
        print(f"tokenized: {tokenized}")
        result = model.generate(input=tokenized, max_new_tokens=64, do_sample=True, **kwargs)
        # print(f"result: {result}")
    return result

def generate_actadd(model, prompts: List[str], layer: int, prompt_add: str, prompt_sub: str, coeff: int, seed: int, sampling_kwargs: Dict[str, Any]) -> List[str]:
    print(f"layer {layer}, coeff {coeff}")
    print(f"prepare_prompts for prompt_add '{prompt_add}' and prompt_sub '{prompt_sub}'")
    prompt_add, prompt_sub = prepare_prompts(prompt_add, prompt_sub, model)
    print(f"prompt_add '{prompt_add}' and prompt_sub '{prompt_sub}' prepared")
    act_add = get_resid_pre(prompt_add, layer, model)
    print(f"act_add.shape: {act_add.shape}, {act_add}")
    act_sub = get_resid_pre(prompt_sub, layer, model)
    print(f"act_sub.shape: {act_sub.shape}, {act_sub}")
    act_diff = act_add - act_sub
    editing_hooks = [(f"blocks.{layer}.hook_resid_pre", lambda resid_pre, hook: ave_hook(resid_pre, hook, act_diff, coeff))]
    results_tensor = hooked_generate(prompts, editing_hooks, seed, model, **sampling_kwargs)
    results_str = model.to_string(results_tensor[:, 1:])
    # results_str_only_generated_text = [results_str[0][len(prompts[0]):]]
    return results_str

In [ ]:
prompt = "I think you're"
act_names = list(range(12)) #l
coeffs = list(range(1, 11)) #c
for act_name in act_names:
  for coeff in coeffs:
    print("layer: ", act_name, " coeffs: ", coeff)
    print(generate_actadd(model_llama, [prompt], act_name, prompt_add, prompt_sub, coeff, SEED, sampling_kwargs))

layer:  0  coeffs:  1


  0%|          | 0/64 [00:00<?, ?it/s]

["I think you're a nice guy.\n\nI'm not saying that I don't like the idea of men being forced to do what they want to do, but I am just saying that I think it's an important part of the game and we need to make sure we have a good understanding of how men are treated in society."]
layer:  1  coeffs:  1


  0%|          | 0/64 [00:00<?, ?it/s]

["I think you're right, and I'm not saying that we should have banned the whole thing.\n\nBut if you want to be clear about what I mean, let me just say that I think it's important to point out a few things. First of all, there are many other things that could have been done differently in this"]
layer:  2  coeffs:  1


...

["I think you're David Bowie.\n\nYou know, I've always been a fan of the artist and he's got a lot of stuff going on in his life. I mean, he's not just playing guitar and singing about things that are going on in his life but he's also doing something very special with it. And there"]
layer:  10  coeffs:  10


  0%|          | 0/64 [00:00<?, ?it/s]

["I think you're David, and I'm not sure how to explain it.\n\nIn the middle of a meeting with a bunch of people in a coffee shop in the city, he said that he'd been working on this for some time. And then I saw him at his desk, looking like he was just finishing up some work"]
layer:  11  coeffs:  10


  0%|          | 0/64 [00:00<?, ?it/s]

["I think you're David, and I'm not sure what to make of your description.\n\nI've been playing with a bunch of different combinations of cards for the past few weeks, and I think it's time to take a look at some ideas that are working well for me. It's important to note that this is not an"]


In [ ]:
# sanity check
from transformer_lens.model_bridge import TransformerBridge

prompt = "I hate you because"
model_llama = model_llama
act_names = list(range(3, 12))
for act_name in act_names:
  resid_pre_original = get_resid_pre(prompt, act_name, model_llama)
  resid_pre_original = resid_pre_original[0]
  print("resid_pre_original.shape: ", resid_pre_original.shape)

  model_mine = TransformerBridge.boot_transformers("gpt2", device="cpu")
  model_mine.enable_compatibility_mode()  # Gemini
  tokens_mine = model_mine.to_tokens(prompt)
  print("token_mine:", tokens_mine)
  logits_mine, cache_mine = model_mine.run_with_cache(tokens_mine, remove_batch_dim=True)
  resid_pre_mine = cache_mine[f"blocks.{act_name}.hook_resid_pre"]
  print("resid_pre_mine.shape: ", resid_pre_mine.shape)
  print("torch.equal()? ", torch.equal(resid_pre_original, resid_pre_mine))
  print("torch.allclose()? ", torch.allclose(resid_pre_original, resid_pre_mine, atol=1e-5))
  break



resid_pre_original.shape:  torch.Size([5, 768])
token_mine: tensor([[50256,    40,  5465,   345,   780]])
resid_pre_mine.shape:  torch.Size([5, 768])
torch.equal()?  True
torch.allclose()?  True


In [ ]:
# sanity check
act_name, coeff = 6, 4

print(generate_actadd(
    model_llama,
     ["The rock hurtled toward the child. The child couldn't get out of the way in time, and so sadly the rock"],
    act_name,
    "I talk about weddings all the time",
    "I never talk about weddings",
    coeff, SEED, sampling_kwargs))

print(generate_actadd(
    model_llama,
     ["The rock hurtled toward the child. The child couldn't get out of the way in time, and so sadly the rock"],
    act_name,
    "I talk about weddings",
    "I never talk about weddings",
    coeff, SEED, sampling_kwargs))

layer 6, coeff 4
prepare_prompts for prompt_add 'I talk about weddings all the time' and prompt_sub 'I never talk about weddings'
tlen(prompt_add):  8  tlen(prompt_sub):  6
prompt_add 'I talk about weddings all the time' and prompt_sub 'I never talk about weddings  ' prepared
act_add.shape: torch.Size([1, 8, 768]), tensor([[[-4.8053, -5.0082, -4.2213,  ..., -3.4563, -3.5408, -4.0276],
         [-1.9010, -1.2025,  1.1621,  ..., -0.2374, -1.0282,  0.8131],
         [-0.7291,  0.4491, -5.9594,  ...,  1.2572,  0.0718, -0.8833],
         ...,
         [-0.8402,  3.9597, -1.1408,  ...,  1.2000, -0.9572,  1.1730],
         [-0.8069,  3.4586, -2.4944,  ...,  0.9294, -2.8059, -1.8582],
         [ 2.6854, -1.9854, -3.1279,  ...,  3.7970, -1.7445, -0.9050]]])
act_sub.shape: torch.Size([1, 8, 768]), tensor([[[-4.8053, -5.0082, -4.2213,  ..., -3.4563, -3.5408, -4.0276],
         [-1.9010, -1.2025,  1.1621,  ..., -0.2374, -1.0282,  0.8131],
         [ 2.3634,  1.9696,  3.6004,  ..., -1.3600,  0.0139

  0%|          | 0/64 [00:00<?, ?it/s]

["The rock hurtled toward the child. The child couldn't get out of the way in time, and so sadly the rock was a symbol of our family's faith in God.\n\nBut this is not the case with people who have been groomed to be a part of a family that is not your own. In fact, it's often an issue for people who are married to other people who are looking for a wedding ring or something"]
layer 6, coeff 4
prepare_prompts for prompt_add 'I talk about weddings' and prompt_sub 'I never talk about weddings'
tlen(prompt_add):  5  tlen(prompt_sub):  6
prompt_add 'I talk about weddings ' and prompt_sub 'I never talk about weddings' prepared
act_add.shape: torch.Size([1, 6, 768]), tensor([[[-4.8053, -5.0082, -4.2213,  ..., -3.4563, -3.5408, -4.0276],
         [-1.9010, -1.2025,  1.1621,  ..., -0.2374, -1.0282,  0.8131],
         [-0.7291,  0.4491, -5.9594,  ...,  1.2573,  0.0718, -0.8833],
         [ 1.3339,  1.2673, -6.0773,  ...,  0.3817,  1.5042,  0.7780],
         [ 6.0509,  2.0652, -1.8830,  ..., -0.

  0%|          | 0/64 [00:00<?, ?it/s]

["The rock hurtled toward the child. The child couldn't get out of the way in time, and so sadly the rock fell on his head. He didn't know what to do, but he did try to help him with it. But he couldn't because he was too busy trying to help his bride.\nWedding Roles\nIt's a big part of our lives for people who are married and live in a family where"]


In [ ]:
def generate_control_text(method,
                          prompt,
                          model,
                          act_name,
                          prompt_add,
                          prompt_sub,
                          coeff,
                          SEED,
                          sampling_kwargs):

    if method == 'actadd':
        while True:
            try:
                prompt_lst = [prompt]
                output = generate_actadd(model,
                                         prompt_lst,
                                         act_name,
                                         prompt_add,
                                         prompt_sub,
                                         coeff,
                                         SEED,
                                         sampling_kwargs)[0]
                break
            except Exception as e:
                error_message = str(e)
                print(f"Generate control text for {method}: something went wrong. Error: {error_message}. Retrying...")
                break

    else:
        raise NotImplementedError

    return output

In [ ]:
# sample from IMDb dataset for NegToPos (0 -> 1)
dataset = load_dataset("stanfordnlp/imdb")
train_dataset = dataset['train']
test_dataset = dataset['test']
merged_dataset = concatenate_datasets([train_dataset, test_dataset])
print("Merged dataset has", merged_dataset.num_rows, "rows")

pos_dataset = merged_dataset.filter(lambda example: example['label'] == 1)
neg_dataset = merged_dataset.filter(lambda example: example['label'] == 0)

model_llama_name = "gpt2"  # "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_llama_name)

def truncate_to_32_tokens(text):
    tokens = tokenizer(text, truncation=True, max_length=32, return_tensors="pt")
    truncated_text = tokenizer.decode(tokens.input_ids[0], skip_special_tokens=True)
    return truncated_text

# pos_dataset = pos_dataset.map(lambda example: {"text": truncate_to_32_tokens(example["text"])})
neg_dataset = neg_dataset.map(lambda example: {"text": truncate_to_32_tokens(example["text"])})

# pos_dataset.to_json(f"{save_dir}/{prompts_setting}/pos_dataset_truncated32.jsonl")
# neg_dataset.to_json(f"{save_dir}/{prompts_setting}/neg_dataset_truncated32.jsonl")

sampled_neg_dataset_indices = random.sample(range(len(neg_dataset)), sample_n)
sampled_neg_dataset = neg_dataset.select(sampled_neg_dataset_indices)

# sampled_neg_dataset.to_json(f"{save_dir}/{prompts_setting}/neg_dataset_sample{sample_n}.jsonl")

Merged dataset has 50000 rows


In [ ]:
for i in range(0,sample_n):
  test = sampled_neg_dataset[i]['text']
  test_tokens = model_llama.to_tokens(test).shape[1]
  print(f'{i}, t={test_tokens}, {test}')

0, t=33, This show seemed to be kinda good. Kyra Sedgwick is an OK actress and I like police series, but somewhere in the production this program went awfully
1, t=33, Oh my... bad clothing, worse synth music and the worst: David Hasselhoff. The 80's are back with vengeance in Witchery, an American-
2, t=33, Is Miike like Chabrol, alternating art with dreck, sometimes confusing the two? Does he match the fifty/fifty rate some claim for Ch
3, t=33, This is by far the worst film I have seen in my entire life. The acting is poor and the storyline is almost incomprehensible. Whether or not you like lights
4, t=33, I bought this because it was $1.99 and Harry Carey was in it and a friend of mine was in it, and for $1.99,
5, t=33, <br /><br />I saw this on the Sci-Fi channel. It came on right after the first one. For some reason this movie kept me
6, t=33, This movie was by far the worst movie I've ever had to endure. I couldn't believe that they tried to pass it off as a serious movie, it
7

# Functions

In [ ]:
###############################################
#             DO NOT RUN                      #
#   TODO: find a 'davinci-002' replacement    #
###############################################
def fluency(prompt, generated_text):
    """Computes fluency using Openai davinci-002 logprobs"""
    response = openai.Completion.create(
      engine='davinci-002',
      prompt=prompt,
      max_tokens=0,
      temperature=0.0,
      logprobs=0,
      echo=True,
    )
    prompt_logprobs = response['choices'][0]['logprobs']['token_logprobs'][1:]

    response = openai.Completion.create(
        engine='davinci-002',
        prompt=generated_text,
        max_tokens=0,
        temperature=0.0,
        logprobs=0,
        echo=True,
    )
    logprobs = response['choices'][0]['logprobs']['token_logprobs'][1:]

    continuation_logprobs = logprobs[len(prompt_logprobs):]
    return np.exp(-np.mean(continuation_logprobs))

In [ ]:
###############################################
#             DO NOT RUN                      #
#   DRIVE IS NOT MOUNTED                      #
###############################################
def write_eval_output_file(outputs, save_dir, prompts_setting, method, act_name, prompt_add, prompt_sub,coeff, num_prompts, note):
    """Writes eval output to a file"""
    def convert(o):
        if isinstance(o, np.float32):
            return float(o)
        raise TypeError
    def clean_and_truncate(input_str, max_length=6):
      cleaned_str = re.sub('[^A-Za-z0-9]+', '', input_str)
      return cleaned_str[:max_length]

    if not os.path.exists(f"{save_dir}/{prompts_setting}"):
        os.makedirs(f"{save_dir}/{prompts_setting}")

    prefix = "gs_" if num_prompts == 50 else ""
    if method == "actadd":
        decode_str = f"l={act_name}_c={coeff}"
        filename = f"{save_dir}/{prompts_setting}/{prefix}NegToPos_{method}_{num_prompts}_{decode_str}_{clean_and_truncate(prompt_add)}_{clean_and_truncate(prompt_sub)}_sentiment_{note}.jsonl"
    else:
        filename = f"{save_dir}/{prompts_setting}/{prefix}{method}_{num_prompts}_{prompt_add}_{prompt_sub}_sentiment_{note}.jsonl"
    with open(filename, "w") as f:
        print(f"c={coeff}, l={act_name}, Saved outputs to {filename}")
        json.dump(outputs, f, default=convert)

In [ ]:
def generate_text_eval(prompts_requested_sampled, method, prompts_setting, model, max_tokens, sampling_kwargs, act_name, prompt_add, prompt_sub, coeff, SEED, note, display=False):
    """Generates completions for the eval set and computes all metrics (tox, perp, rel)"""

    # model used for relevance - embeddings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_rel = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2").to(device)

    # model used for success metric
    sentiment_analysis = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")

    generations = []
    outputs = []
    generated_text_all = []

    for elem in tqdm(prompts_requested_sampled):
        print(f"For c={coeff}, l={act_name}")
        if len(elem['text']) < 3:
            continue

        generated_text = generate_control_text(method=method,
                                                prompt=elem['text'],
                                                model=model,
                                                act_name=act_name,
                                                prompt_add=prompt_add,
                                                prompt_sub=prompt_sub,
                                                coeff=coeff,
                                                SEED=SEED,
                                                sampling_kwargs=sampling_kwargs)

        # if generated_text.endswith('</s>'):
        #     if not generated_text:
        #       continue
        #     else:
        #       generated_text = generated_text[:-4].strip()
        # if generated_text.startswith('</s>'):
        #       generated_text = generated_text[4:].strip()

        # Success
        continuation_sentiment_analysis = sentiment_analysis(generated_text[len(elem['text']):].strip())

        if continuation_sentiment_analysis[0]['label'] == 'POSITIVE':
            continuation_label = 1
        elif continuation_sentiment_analysis[0]['label'] == 'NEGATIVE':
            continuation_label = 0
        else:
            continuation_label = 'check_again'

        # Fluency
        # davinci_continuation_perplexity = fluency(elem['text'], generated_text)

        # Relevance
        embeddings_prompt = model_rel.encode(elem['text'])
        embeddings_continuation = model_rel.encode(generated_text[len(elem['text']):].strip())
        similarity = cosine_similarity(embeddings_prompt.reshape(1, -1), embeddings_continuation.reshape(1, -1))[0][0]

        if display:
            print("Prompt:", elem['text'], "\n")
            print(f"Generated Text by {method}:", generated_text[len(elem['text']):].strip(), "\n")
            print(f"Cont Sent: {continuation_label}, Prompt Sent(label):{elem['label']}, Relevance: {similarity}""\n\n=====\n")
            # print(f"Cont Sent: {continuation_label}, Prompt Sent(label):{elem['label']}, Fluency:{davinci_continuation_perplexity}, Relevance: {similarity}""\n\n=====\n")

        generations.append(generated_text)

        generated_text_all.append(generated_text[len(elem['text']):].strip())

        outputs.append({"content": generated_text,
                        "prompt": elem['text'],
                        "continuation": generated_text[len(elem['text']):].strip(),
                        "prompt_label": elem['label'],
                        "continuation_label": continuation_label,
                        "continuation_sentiment_analysis": continuation_sentiment_analysis,
                        # "davinci_continuation_perplexity": davinci_continuation_perplexity,
                        "relevance_similarity": similarity})
    if len(prompts_requested_sampled) >= 10:
        num_prompts = len(prompts_requested_sampled)
        print("write_eval_output_file")
        # write_eval_output_file(outputs,save_dir, prompts_setting, method, act_name, prompt_add, prompt_sub,coeff, num_prompts, note)

    return generated_text_all, outputs


In [ ]:
# we perform this fix to the function in transformer_lens for version 1.17.0: https://github.com/neelnanda-io/TransformerLens/pull/578
import transformer_lens
from typing import Optional, Union, Tuple, Callable, List, cast
from functools import partial
from transformer_lens.hook_points import NamesFilter
from transformer_lens.utils import Slice, SliceInput

def get_caching_hooks(
        self,
        names_filter: NamesFilter = None,
        incl_bwd: bool = False,
        device=None,
        remove_batch_dim: bool = False,
        cache: Optional[dict] = None,
        pos_slice: Union[Slice, SliceInput] = None,
    ) -> Tuple[dict, list, list]:
        """Creates hooks to cache activations. Note: It does not add the hooks to the model.

        Args:
            names_filter (NamesFilter, optional): Which activations to cache. Can be a list of strings (hook names) or a filter function mapping hook names to booleans. Defaults to lambda name: True.
            incl_bwd (bool, optional): Whether to also do backwards hooks. Defaults to False.
            device (_type_, optional): The device to store on. Keeps on the same device as the layer if None.
            remove_batch_dim (bool, optional): Whether to remove the batch dimension (only works for batch_size==1). Defaults to False.
            cache (Optional[dict], optional): The cache to store activations in, a new dict is created by default. Defaults to None.

        Returns:
            cache (dict): The cache where activations will be stored.
            fwd_hooks (list): The forward hooks.
            bwd_hooks (list): The backward hooks. Empty if incl_bwd is False.
        """
        if cache is None:
            cache = {}

        if not isinstance(pos_slice, Slice):
            if isinstance(
                pos_slice, int
            ):  # slicing with an int collapses the dimension so this stops the pos dimension from collapsing
                pos_slice = [pos_slice]
            pos_slice = Slice(pos_slice)

        if names_filter is None:
            names_filter = lambda name: True
        elif isinstance(names_filter, str):
            filter_str = names_filter
            names_filter = lambda name: name == filter_str
        elif isinstance(names_filter, list):
            filter_list = names_filter
            names_filter = lambda name: name in filter_list
        self.is_caching = True

        # mypy can't seem to infer this
        names_filter = cast(Callable[[str], bool], names_filter)

        def save_hook(tensor, hook, is_backward=False):
            hook_name = hook.name
            if is_backward:
                hook_name += "_grad"
            resid_stream = tensor.detach().to(device)
            if remove_batch_dim:
                resid_stream = resid_stream[0]

            # for attention heads the pos dimension is the third from last
            if (
                hook.name.endswith("hook_q")
                or hook.name.endswith("hook_k")
                or hook.name.endswith("hook_v")
                or hook.name.endswith("hook_z")
                or hook.name.endswith("hook_result")
            ):
                pos_dim = -3
            else:
                # for all other components the pos dimension is the second from last
                # including the attn scores where the dest token is the second from last
                pos_dim = -2

            if (
                tensor.dim() >= -pos_dim
            ):  # check if the residual stream has a pos dimension before trying to slice
                resid_stream = pos_slice.apply(resid_stream, dim=pos_dim)
            cache[hook_name] = resid_stream

        fwd_hooks = []
        bwd_hooks = []
        for name, hp in self.hook_dict.items():
            if names_filter(name):
                fwd_hooks.append((name, partial(save_hook, is_backward=False)))
                if incl_bwd:
                    bwd_hooks.append((name, partial(save_hook, is_backward=True)))

        return cache, fwd_hooks, bwd_hooks

# Replace the original get_caching_hooks function
transformer_lens.hook_points.HookedRootModule.get_caching_hooks = get_caching_hooks

/tmp/ipykernel_793/1388720989.py:6: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens.utils import Slice, SliceInput
/tmp/ipykernel_793/1388720989.py:93: DeprecationWarning: Importing HookedRootModule from transformer_lens.hook_points is deprecated and will be removed in a future release. Import it from transformer_lens (preferred) or transformer_lens.HookedRootModule instead.
  transformer_lens.hook_points.HookedRootModule.get_caching_hooks = get_caching_hooks


# Run Sentiment Experiment

In [ ]:
sentiment_samples_paths = [f"{save_dir}/{prompts_setting}/neg_dataset_sample{sample_n}.jsonl"]

In [ ]:
#dataset
prompts_requested_sampled = {}
for i, path in enumerate(sentiment_samples_paths):
    dataset_num = i+1
    filename = sentiment_samples_paths[dataset_num-1]
    print(filename)
    note = f"dataset{dataset_num}"
    data_random = []
    with open(filename, "r") as f:
        for line in f:
            data_random.append(json.loads(line))
    prompts_requested_sampled[dataset_num] = data_random
    print(f"First lines of {note}: {prompts_requested_sampled[dataset_num][:3]}")

/content/drive/MyDrive/actadd_reb/llama3_sentiment/neg_dataset_sample10.jsonl
First lines of dataset1: [{'text': 'Amando DeOssorio was never one to let a lack of budget get in the way of telling one of his stories. His "Blind Dead', 'label': 0}, {'text': 'I can sit through this movie once, but I doubt I could make it through a second time. Mildly entertaining mainly for the physical presence of Lindsay L', 'label': 0}, {'text': 'Revenge of the Sith starts out with a long action sequence that is impressive without being terribly exciting, then gets really boring for the next hour and fifteen minutes', 'label': 0}]


In [ ]:
# Run completions
n = sample_n

max_retries = 4
retry_count = 0

for i, dataset in enumerate(prompts_requested_sampled):
    dataset_num = i+1
    note = f"dataset{dataset_num}"
    print(note, len(prompts_requested_sampled[dataset][:n]), prompts_requested_sampled[dataset][:n])
    while True:
      try:
        generations, outputs = generate_text_eval(prompts_requested_sampled=prompts_requested_sampled[dataset][:n],
                                          method=method,
                                          prompts_setting=prompts_setting,
                                          model=model_llama,
                                          max_tokens=32, # dummy, sentiment is 64 by default
                                          sampling_kwargs=sampling_kwargs,
                                          act_name=act_name,
                                          prompt_add=prompt_add,
                                          prompt_sub=prompt_sub,
                                          coeff=coeff,
                                          SEED=SEED,
                                          note=note,
                                          display=display)
        break
      except Exception as e:
        print(f"Error communicating with OpenAI: {e}")

        if retry_count >= max_retries:
            raise Exception("Maximum number of retries exceeded")

        time.sleep(5)  # Wait for 5 seconds before retrying

dataset1 10 [{'text': 'Amando DeOssorio was never one to let a lack of budget get in the way of telling one of his stories. His "Blind Dead', 'label': 0}, {'text': 'I can sit through this movie once, but I doubt I could make it through a second time. Mildly entertaining mainly for the physical presence of Lindsay L', 'label': 0}, {'text': 'Revenge of the Sith starts out with a long action sequence that is impressive without being terribly exciting, then gets really boring for the next hour and fifteen minutes', 'label': 0}, {'text': 'This movie was probably the worst movie I have ever seen. Here are the things that immediately jump out at me: 1. The woods were more like', 'label': 0}, {'text': 'Abysmal with a capital "A". This has got to be one of, if not THE, unfunniest show on TV right now. I', 'label': 0}, {'text': 'I saw this film in the movie theater. I was taking classes at the Second City Chicago and of course the buzz of this movie was intense. It is', 'label': 0}, {'text': 'Th

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Prompt: This stinker is in mystifyingly frequent rotation on one channel here, and I've found myself watching in horror again and again. The script is like 

Generated Text by actadd: a love letter to Love 2 Hate, with the same type of love song that was so loved by Love 2 Love. The movie is so far from being love 2 love that it's actually more like a love 2 hate. It's all about loving your lover but not really loving your lover because you 

Cont Sent: 0, Prompt Sent(label):0, Fluency:13.971499531606131, Relevance: 0.26090100407600403

=====

For c=12, l=17


  0%|          | 0/64 [00:00<?, ?it/s]

100%|██████████| 10/10 [01:20<00:00,  8.01s/it]

Prompt: This is a really mediocre film in the vein of "Buckaroo Banzai." The cast runs around like "Mad Max" wannabes, 

Generated Text by actadd: but the love-lo loved-lo love-lo love-lo Love 3: Love Love 2: Love 1: Love Love God is one lover. He loves you and me. He loves us so much that he sent his only son to die for our freedom. This was true love. It was not just a mere 

Cont Sent: 1, Prompt Sent(label):0, Fluency:24.388278990821032, Relevance: 0.06619984656572342

=====



c=12, l=17, Saved outputs to /content/drive/MyDrive/actadd_reb/llama3_sentiment/NegToPos_actadd_10_l=17_c=12_Love_Hate_sentiment_dataset1.jsonl


# Analyse sentiment results

In [ ]:
def analyze_sentiment(outputs):
    results_list = []
    for fname in outputs:
        with open(fname, 'r') as f:
            lines = json.load(f)

        match = re.search(r'l=(\d+)_c=(\d+)', fname)
        if match:
            l_value, c_value = match.groups()
        else:
            print(f"Could not extract l and c values from {fname}")
            continue

        prompt_label = []
        cont_label = []
        ppl = []
        rel = []
        total = len(lines)
        for line in lines:
            prompt_label.append(line['prompt_label'])
            cont_label.append(line['continuation_label'])
            ppl.append(line['davinci_continuation_perplexity'])
            rel.append(line['relevance_similarity'])

        label_agree_count = sum(1 for prompt, cont in zip(prompt_label, cont_label) if prompt != cont)
        success = label_agree_count / total if total > 0 else 0
        avg_ppl = sum(ppl) / total
        avg_rel = sum(rel) / total

        print("Statistics of", fname)
        print(f"    Sample size: {total}")
        print(f"    Success: {success}")
        print(f"    Average perplexity of continuations: {avg_ppl}\n")
        print(f"    Average relevance of continuations: {avg_rel}\n")

        results_list.append({
            'Filename': fname,
            'L': l_value,
            'C': c_value,
            'Sample Size': total,
            'Success': success,
            'Average Perplexity of Continuations': avg_ppl,
            'Average Relevance of Continuations': avg_rel
        })

    return results_list


In [ ]:
array_filename = ['NegToPos_actadd_10_l=17_c=12_Love_Hate_sentiment_dataset1.jsonl']
array_fullpath = [f"{save_dir}/{prompts_setting}/" + fn for fn in array_filename]

get_sent_results = analyze_sentiment(array_fullpath)

Statistics of /content/drive/MyDrive/actadd_reb/llama3_sentiment/NegToPos_actadd_10_l=17_c=12_Love_Hate_sentiment_dataset1.jsonl
    Sample size: 10
    Success: 0.6
    Average perplexity of continuations: 16.547487150893748

    Average relevance of continuations: 0.18052267655730247



In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
########################################
# OpenAI
########################################

# OPENAI_API_KEY = ''
# openai.api_key = OPENAI_API_KEY

In [ ]:
# gemini trying to fix `from sentence_transformers import SentenceTransformer` error
print('Installing FFmpeg and torchcodec... This may take a few moments.')
!apt-get update
!apt-get install -y ffmpeg
# ffmpeg 7:4.4.2-0ubuntu0.22.04.1
!pip install torchcodec # Install torchcodec after FFmpeg and PyTorch are correctly set up.
# Update dynamic linker run-time bindings
!ldconfig /usr/local/lib/
print('FFmpeg and torchcodec installation complete.')

In [ ]:
########################################
# HuggingFace log in for LLAMA
########################################
# !huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful
